Explore Spotify Releases.

- Does every entry have a title, artist, release date and ISRC?
- What are the earliest and latest release dates?
- How many fall within the study window?
- Are any recordings listed more than once?
- Do a few artists account for many of the results?

## DataFrame reference

Reference for the current dataset after running the notebook from top to bottom. Update this table when the data or preparation steps change.

| DataFrame | What one row represents | Contents / purpose | Created from |
|---|---|---|---|
| `tracks_df` | One Spotify track entry | All 100 entries, original nested metadata, and extracted ISRC | JSON `data["tracks"]` |
| `album_df` | Album information for one track entry | Expanded album fields and converted release dates; 100 rows, not necessarily unique albums | `tracks_df["album"]` |
| `filtered_album_df` | Album information for one eligible track | 68 entries within January 1–September 1, 2026, inclusive; original indexes retained | Date-filtered `album_df` |
| `filtered_tracks_df` | One eligible track entry | The matching 68 tracks with nested metadata and ISRCs; converted dates remain in the album table | `tracks_df`, selected using `filtered_album_df.index` |
| `track_artist_df` | One track–artist credit | 143 rows with `track_id`, `artist_id`, and `artist_name`; 133 unique artist IDs | Expanded artist lists from `filtered_tracks_df` |

**Rebuilding tables:** After changing an earlier preparation step, rerun the downstream cells—or restart the kernel and Run All. Existing derived DataFrames do not update automatically.

**Index alignment:** The track and album tables share their original row indexes. Preserve those indexes until the matching filtered tracks have been selected. The artist-credit table uses a new index and retains `track_id` to link credits back to tracks.


## 1. Import libraries

In [1]:
import json
import pandas as pd

## 2. Load data and create a DataFrame

In [2]:
# import json file, using utf-8 for translating a file's
# bytes to characters, save as data
with open("data/100_edm_tracks.json", encoding="utf-8") as file: 
    data = json.load(file)

tracks_df = pd.DataFrame(data["tracks"])
tracks_df.head()


,album,artists,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,is_local,is_playable,name,track_number,type,uri
0,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,127846,False,{'isrc': 'QT3F92642024'},{'spotify': 'https://open.spotify.com/track/2s...,https://api.spotify.com/v1/tracks/2sxYN584YAgn...,2sxYN584YAgnRwnmIQF0Ye,False,True,CRECIENDO DIARIAMENTE - Afro House,3,track,spotify:track:2sxYN584YAgnRwnmIQF0Ye
1,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,162804,False,{'isrc': 'DK4YA2600951'},{'spotify': 'https://open.spotify.com/track/0f...,https://api.spotify.com/v1/tracks/0fS2ZLRBEehM...,0fS2ZLRBEehMSTUXyy78mM,False,True,Losing My Religion,1,track,spotify:track:0fS2ZLRBEehMSTUXyy78mM
2,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,133250,False,{'isrc': 'QZQAY2668235'},{'spotify': 'https://open.spotify.com/track/2P...,https://api.spotify.com/v1/tracks/2PuZF8KqwkGN...,2PuZF8KqwkGN9eEatQ1xDs,False,True,Mi Chico - Josh Fawaz Remix,3,track,spotify:track:2PuZF8KqwkGN9eEatQ1xDs
3,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,183870,False,{'isrc': 'QZTPX2680490'},{'spotify': 'https://open.spotify.com/track/65...,https://api.spotify.com/v1/tracks/65cd1kUvEU9Q...,65cd1kUvEU9Q0PmYJRc6Uy,False,True,Carne y Tambor,1,track,spotify:track:65cd1kUvEU9Q0PmYJRc6Uy
4,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,196500,False,{'isrc': 'AUNV02600073'},{'spotify': 'https://open.spotify.com/track/7v...,https://api.spotify.com/v1/tracks/7vY4hbChNHJu...,7vY4hbChNHJudQQpPlnL5M,False,True,Deeper Love,1,track,spotify:track:7vY4hbChNHJudQQpPlnL5M


## 3. Initial data inspection

In [3]:
# rows, columns, datatypes
tracks_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   album          100 non-null    object
 1   artists        100 non-null    object
 2   disc_number    100 non-null    int64 
 3   duration_ms    100 non-null    int64 
 4   explicit       100 non-null    bool  
 5   external_ids   100 non-null    object
 6   external_urls  100 non-null    object
 7   href           100 non-null    str   
 8   id             100 non-null    str   
 9   is_local       100 non-null    bool  
 10  is_playable    100 non-null    bool  
 11  name           100 non-null    str   
 12  track_number   100 non-null    int64 
 13  type           100 non-null    str   
 14  uri            100 non-null    str   
dtypes: bool(3), int64(3), object(4), str(5)
memory usage: 9.8+ KB


In [4]:
# count missing values in top-level columns
tracks_df.isna().sum()

album            0
artists          0
disc_number      0
duration_ms      0
explicit         0
external_ids     0
external_urls    0
href             0
id               0
is_local         0
is_playable      0
name             0
track_number     0
type             0
uri              0
dtype: int64

The dataset contains 100 entries and 15 columns. No top-level missing values were detected. Nested fields still need inspection.

## 4. Prepare fields for analysis

In [5]:
# create a new column with isrc code, using the external_ids dictionary
tracks_df["isrc"] = tracks_df["external_ids"].apply(
    lambda x: x.get("isrc") if isinstance(x, dict) else None
)

# check for null values in the new isrc column
tracks_df["isrc"].isna().sum()

np.int64(0)

In [6]:
# check for empty nested fields next
# Expand album dictionaries into a table
album_df = pd.json_normalize(tracks_df["album"])

# Count missing values in the resulting columns
album_df.isna().sum()

album_type                0
artists                   0
href                      0
id                        0
images                    0
is_playable               0
name                      0
release_date              0
release_date_precision    0
total_tracks              0
type                      0
uri                       0
external_urls.spotify     0
dtype: int64

In [7]:
# check whether release dates are usable for analysis
print(f"Release date value counts:\n", album_df[["release_date"]].value_counts(), sep='')

Release date value counts:
release_date
2026-03-06      5
2026-09-25      5
2026-05-29      5
2026-10-30      4
2026-06-08      3
2026-09-18      3
2026-08-21      3
2026-10-09      3
2026-05-08      3
2026-03-13      3
2026-08-28      3
2026-04-03      3
2026-10-16      3
2026-10-02      3
2026-09-04      2
2026-10-24      2
2026-02-06      2
2026-05-15      2
2026-01-23      2
2026-03-26      2
2026-03-27      2
2026-07-24      2
2026-06-26      2
2026-09-11      2
2026-04-17      2
2026-07-03      2
2026-05-22      2
2026-05-01      2
2026-01-15      1
2026-06-19      1
2026-10-25      1
2026-06-14      1
2026-02-27      1
2026-10-15      1
2026-04-24      1
2026-06-12      1
2026-08-07      1
2026-04-10      1
2026-06-05      1
2026-10-17      1
2026-08-26      1
2026-04-29      1
2026-07-18      1
2026-10-26      1
2026-01-29      1
2026-01-09      1
2026-07-14      1
2026-10-23      1
2026-03-05      1
2026-01-24      1
2026-04-28      1
Name: count, dtype: int64


In [8]:
# sanity check
album_df.head(5)

,album_type,artists,href,id,images,is_playable,name,release_date,release_date_precision,total_tracks,type,uri,external_urls.spotify
0,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5jKfzHgw2kUj...,5jKfzHgw2kUj2GgzZZyXDV,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,POR MáS DíAS DE BIEN,2026-06-08,day,3,album,spotify:album:5jKfzHgw2kUj2GgzZZyXDV,https://open.spotify.com/album/5jKfzHgw2kUj2Gg...
1,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2tBXlHUu40fy...,2tBXlHUu40fyP9eKw13D1L,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Losing My Religion,2026-09-04,day,1,album,spotify:album:2tBXlHUu40fyP9eKw13D1L,https://open.spotify.com/album/2tBXlHUu40fyP9e...
2,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5BskxQl78jot...,5BskxQl78jotslv7DMPW4p,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Mi Chico (Josh Fawaz Remix),2026-10-24,day,3,album,spotify:album:5BskxQl78jotslv7DMPW4p,https://open.spotify.com/album/5BskxQl78jotslv...
3,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/4UCEjU56lQmW...,4UCEjU56lQmWaAIc6bxxea,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Carne y Tambor,2026-02-06,day,1,album,spotify:album:4UCEjU56lQmWaAIc6bxxea,https://open.spotify.com/album/4UCEjU56lQmWaAI...
4,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2ZwcQ15i7nzK...,2ZwcQ15i7nzKnXh5zIkhhO,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Deeper Love,2026-05-15,day,1,album,spotify:album:2ZwcQ15i7nzKnXh5zIkhhO,https://open.spotify.com/album/2ZwcQ15i7nzKnXh...


In [9]:
# validate dates are in a usable format
album_df["release_date"] = pd.to_datetime(
    album_df["release_date"], 
    errors="coerce")

# check for missing values in the release_date column
print(f"Missing Values: ", album_df["release_date"].isna().sum())

#
print(f"Data type: ", album_df["release_date"].dtypes)


Missing Values:  0
Data type:  datetime64[us]


## 5. Select the study window

In [10]:
# filter dates to only inlude January 1, 2026 - September 1, 2026
# make working copy of the filtered dataframe to avoid SettingWithCopyWarning

filtered_album_df = album_df[(
    album_df["release_date"] >= "2026-01-01") & 
    (album_df["release_date"] <= "2026-09-01")].copy()

filtered_album_df.head(5)

,album_type,artists,href,id,images,is_playable,name,release_date,release_date_precision,total_tracks,type,uri,external_urls.spotify
0,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/5jKfzHgw2kUj...,5jKfzHgw2kUj2GgzZZyXDV,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,POR MáS DíAS DE BIEN,2026-06-08,day,3,album,spotify:album:5jKfzHgw2kUj2GgzZZyXDV,https://open.spotify.com/album/5jKfzHgw2kUj2Gg...
3,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/4UCEjU56lQmW...,4UCEjU56lQmWaAIc6bxxea,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Carne y Tambor,2026-02-06,day,1,album,spotify:album:4UCEjU56lQmWaAIc6bxxea,https://open.spotify.com/album/4UCEjU56lQmWaAI...
4,single,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/2ZwcQ15i7nzK...,2ZwcQ15i7nzKnXh5zIkhhO,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,Deeper Love,2026-05-15,day,1,album,spotify:album:2ZwcQ15i7nzKnXh5zIkhhO,https://open.spotify.com/album/2ZwcQ15i7nzKnXh...
5,album,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/1aHWSGAetL13...,1aHWSGAetL13Kkl2amjrnQ,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,SOMA,2026-03-06,day,21,album,spotify:album:1aHWSGAetL13Kkl2amjrnQ,https://open.spotify.com/album/1aHWSGAetL13Kkl...
6,album,[{'external_urls': {'spotify': 'https://open.s...,https://api.spotify.com/v1/albums/1aHWSGAetL13...,1aHWSGAetL13Kkl2amjrnQ,"[{'height': 640, 'width': 640, 'url': 'https:/...",True,SOMA,2026-03-06,day,21,album,spotify:album:1aHWSGAetL13Kkl2amjrnQ,https://open.spotify.com/album/1aHWSGAetL13Kkl...


In [11]:
# confirm the count of rows in the filtered dataframe is 
# less than the original dataframe
print(f"Original DataFrame rows: {len(album_df)}")
print(f"Filtered DataFrame rows: {len(filtered_album_df)}")

Original DataFrame rows: 100
Filtered DataFrame rows: 68


In [12]:
# apply the same date selection to the original tracks_df dataframe, 
# using the index of the filtered_album_df
# make working copy to save

filtered_tracks_df = tracks_df.loc[filtered_album_df.index].copy()

print(f"Filtered Tracks DataFrame rows: {len(filtered_tracks_df)}")

# check row 7 of both dataframes to confirm they are the same
print(f"Filtered Album DataFrame row 7:\n", filtered_album_df["name"].iloc[7])
print(f"Filtered Tracks DataFrame row 7:\n", filtered_tracks_df["name"].iloc[7])

filtered_tracks_df.head(5)

Filtered Tracks DataFrame rows: 68
Filtered Album DataFrame row 7:
 Memeza, Pt. 1
Filtered Tracks DataFrame row 7:
 Memeza, Pt. 1


,album,artists,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,is_local,is_playable,name,track_number,type,uri,isrc
0,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,127846,False,{'isrc': 'QT3F92642024'},{'spotify': 'https://open.spotify.com/track/2s...,https://api.spotify.com/v1/tracks/2sxYN584YAgn...,2sxYN584YAgnRwnmIQF0Ye,False,True,CRECIENDO DIARIAMENTE - Afro House,3,track,spotify:track:2sxYN584YAgnRwnmIQF0Ye,QT3F92642024
3,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,183870,False,{'isrc': 'QZTPX2680490'},{'spotify': 'https://open.spotify.com/track/65...,https://api.spotify.com/v1/tracks/65cd1kUvEU9Q...,65cd1kUvEU9Q0PmYJRc6Uy,False,True,Carne y Tambor,1,track,spotify:track:65cd1kUvEU9Q0PmYJRc6Uy,QZTPX2680490
4,"{'album_type': 'single', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,1,196500,False,{'isrc': 'AUNV02600073'},{'spotify': 'https://open.spotify.com/track/7v...,https://api.spotify.com/v1/tracks/7vY4hbChNHJu...,7vY4hbChNHJudQQpPlnL5M,False,True,Deeper Love,1,track,spotify:track:7vY4hbChNHJudQQpPlnL5M,AUNV02600073
5,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,1,206000,False,{'isrc': 'GXBAV2567659'},{'spotify': 'https://open.spotify.com/track/53...,https://api.spotify.com/v1/tracks/53IpJlqruSht...,53IpJlqruShtCrrs09bqH1,False,True,AMADO,10,track,spotify:track:53IpJlqruShtCrrs09bqH1,GXBAV2567659
6,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,1,236000,False,{'isrc': 'QT5M22515698'},{'spotify': 'https://open.spotify.com/track/1y...,https://api.spotify.com/v1/tracks/1y2plKQTw6Py...,1y2plKQTw6PyGHxTmVJpuM,False,True,ALGORA [MAQUINA],17,track,spotify:track:1y2plKQTw6PyGHxTmVJpuM,QT5M22515698


## 6. Inspect duplicate records

In [13]:
# check for duplicates in the filtered tracks dataframe
print(f"Duplicate rows in filtered_tracks_df: ", filtered_tracks_df["isrc"].duplicated().sum())

Duplicate rows in filtered_tracks_df:  0


In [17]:
# inspect one artist name to see if it is a string or a list
print(f"Artist inspection: ", filtered_tracks_df["artists"].iloc[0])

Artist inspection:  [{'external_urls': {'spotify': 'https://open.spotify.com/artist/204dOGG0fctk51d79aqtJG'}, 'href': 'https://api.spotify.com/v1/artists/204dOGG0fctk51d79aqtJG', 'id': '204dOGG0fctk51d79aqtJG', 'name': 'Cokito Deep', 'type': 'artist', 'uri': 'spotify:artist:204dOGG0fctk51d79aqtJG'}]


In [21]:
# create a new DataFrame with one row per track-artist credit
track_artist_df = (
    filtered_tracks_df[["id", "artists"]]
    .explode("artists") # turns each item in a list to a separate row
    .reset_index(drop=True)
)
track_artist_df.head(5)
track_artist_df.loc[0, "artists"]

{'external_urls': {'spotify': 'https://open.spotify.com/artist/204dOGG0fctk51d79aqtJG'},
 'href': 'https://api.spotify.com/v1/artists/204dOGG0fctk51d79aqtJG',
 'id': '204dOGG0fctk51d79aqtJG',
 'name': 'Cokito Deep',
 'type': 'artist',
 'uri': 'spotify:artist:204dOGG0fctk51d79aqtJG'}

In [22]:
# extract artist ID and names into their own columns
track_artist_df["artist_id"] = track_artist_df["artists"].apply(
    lambda artist: artist.get("id")
)

track_artist_df["artist_name"] = track_artist_df["artists"].apply(
    lambda artist: artist.get("name")
)

track_artist_df.head(5)

,id,artists,artist_id,artist_name
0,2sxYN584YAgnRwnmIQF0Ye,{'external_urls': {'spotify': 'https://open.sp...,204dOGG0fctk51d79aqtJG,Cokito Deep
1,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,00CMSJdbf36zOzKB3z8JrR,Victor Cardenas
2,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,14ASSasIb42tG16ym4kVfh,Exotic
3,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,62mc7565AIVwculibZaNOZ,Konig Pry
4,65cd1kUvEU9Q0PmYJRc6Uy,{'external_urls': {'spotify': 'https://open.sp...,4IrirmbSt5yUzaeQ0wvO0F,Ricardo RV


In [23]:
# rename track ID column and keep the three fields
track_artist_df = track_artist_df.rename(columns={"id": "track_id"})[["track_id", "artist_id", "artist_name"]]
track_artist_df[["track_id", "artist_id", "artist_name"]]
track_artist_df.head(5)

,track_id,artist_id,artist_name
0,2sxYN584YAgnRwnmIQF0Ye,204dOGG0fctk51d79aqtJG,Cokito Deep
1,65cd1kUvEU9Q0PmYJRc6Uy,00CMSJdbf36zOzKB3z8JrR,Victor Cardenas
2,65cd1kUvEU9Q0PmYJRc6Uy,14ASSasIb42tG16ym4kVfh,Exotic
3,65cd1kUvEU9Q0PmYJRc6Uy,62mc7565AIVwculibZaNOZ,Konig Pry
4,65cd1kUvEU9Q0PmYJRc6Uy,4IrirmbSt5yUzaeQ0wvO0F,Ricardo RV


In [25]:
# count unique artists in the filtered tracks dataframe
# nuunique() returns the number of unique values in a Series
unique_artists_count = track_artist_df["artist_id"].nunique()
unique_artists_count

133